# Grid Trotter costs

The kinetic and Coulomb terms act on a first-quantized real-space grid.
This notebook follows distance scaling, piecewise inverse-square-root
interpolation, the Newton correction, QROM intervals and the cost comparisons.

Adapted from [Qualtran v0.7.0](https://github.com/quantumlib/Qualtran/tree/bcb17bb25360ed68ce3bc084645db3e929799389/qualtran/bloqs/chemistry/trotter/grid_ham/trotter_costs.ipynb), Copyright 2023 Google LLC, Apache-2.0.

## Real-time evolution

For Hamiltonian $H=T+V$, a first-order step applies
$e^{-iT\delta t}e^{-iV\delta t}$. Repeating it $m$ times with
$\delta t=t/m$ approaches $e^{-iHt}$. The local error is second order in
$\delta t$ and depends on the commutator of the kinetic and Coulomb terms.
This tutorial estimates the terms individually.

## First-quantized grid

For 61 points on a line of length 15, the spacing is $\Delta=1/4$.
Distinct, nonzero distances on the integer grid are $d=1,\ldots,60$.
The physical distance is $r=\Delta d$ and the Coulomb factor is $1/(\Delta d)$.
Coincident points are excluded from this plot.

In [ ]:
from io import BytesIO

from IPython.display import Image, Markdown, SVG, display
import matplotlib.pyplot as plt
import numpy as np

def show_figure(figure):
    with BytesIO() as buffer:
        figure.savefig(buffer, format="png")
        plt.close(figure)
        display(Image(data=buffer.getvalue()))


half_grid = 30
integer_grid = np.arange(-half_grid, half_grid + 1)
spacing = 15.0 / (integer_grid.size - 1)
physical_grid = spacing * integer_grid
left, right = np.triu_indices(integer_grid.size, k=1)
integer_distances = np.unique(np.abs(integer_grid[left] - integer_grid[right]))
distances = np.unique(np.abs(physical_grid[left] - physical_grid[right]))
assert spacing == 0.25
assert np.ptp(physical_grid) == 15
assert np.array_equal(integer_distances, np.arange(1, 61))
np.testing.assert_allclose(distances, integer_distances * spacing)
np.testing.assert_allclose(1 / distances, (1 / integer_distances) / spacing)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.4), layout="constrained")
for grid, axis, label in zip(
    (distances, integer_distances), axes, ("Physical distance", "Integer distance"), strict=True
):
    dense = np.linspace(grid[0], grid[-1], 100)
    axis.plot(dense, 1 / dense, label="1 / distance")
    axis.scatter(grid, 1 / grid, s=9, label="Grid points")
    axis.set(xlabel=label, ylabel="Coulomb factor")
    axis.legend()
show_figure(fig)

## Pair phase and workspace

For $\eta$ electrons, the system has shape $(\eta,3)$ with
$1+\lceil\log_2(\mathrm{grid\ points})\rceil$ qubits per axis in the locked
cost convention. Each electron pair contributes a phase
$e^{-i\delta t/r_{ij}}$. Compute the squared coordinate difference, estimate
its inverse square root, apply the phase, and release the work by uncomputation.
The native zero-distance lookup uses zero; the physical Coulomb potential
is singular there.

## Interpolation and Newton refinement

Let $x=r_{ij}^2$. The QROM supplies coefficients for a cubic approximation
to $x^{-1/2}$. The two cubics are centered at $1$ and $3/2$.
A Newton step uses $y_1=y_0(3+\delta-xy_0^2)/2$. The two interval
corrections reproduce the locked tutorial.

In [ ]:
from flagquantum.algorithms.primitives.grid_cubic_inverse_sqrt import _DELTA_HIGH, _DELTA_LOW
from flagquantum.algorithms.primitives.grid_inverse_sqrt import _HIGH_DECIMAL, _LOW_DECIMAL

coefficients = tuple(tuple(map(float, row)) for row in (_LOW_DECIMAL, _HIGH_DECIMAL))
deltas = tuple(map(float, (_DELTA_LOW, _DELTA_HIGH)))


def cubic(x, region, power=0):
    a, b, c, d = coefficients[region]
    scale = 2**power
    offset = x / scale - (1.0, 1.5)[region]
    return (a - offset * (b - offset * (c - offset * d))) / np.sqrt(scale)


def newton(x, initial, region):
    return 0.5 * initial * (3 + deltas[region] - x * initial**2)


fig, axes = plt.subplots(1, 2, figsize=(9, 3.4), layout="constrained")
for region, (start, stop) in enumerate(((1.0, 1.5), (1.5, 2.0))):
    x = np.linspace(start, stop, 10)
    expected = 1 / np.sqrt(x)
    initial = cubic(x, region)
    updated = newton(x, initial, region)
    assert np.max(np.abs(expected - updated)) < 1e-8
    axes[0].semilogy(x, np.abs(expected - initial), "o", label=f"Region {region + 1}")
    axes[1].semilogy(x, np.abs(expected - updated), "o", label=f"Region {region + 1}")
for title, axis in zip(("Cubic", "After Newton"), axes, strict=True):
    axis.set(title=title, xlabel="Normalized squared distance", ylabel="Absolute error")
    axis.legend()
show_figure(fig)

## Scaling and QROM regions

For $2^k\le x<2^{k+1}$, divide the argument by $2^k$ and the result by
$2^{k/2}$. For $k\ge2$, the bit immediately below the leading bit selects
the lower or upper interval. The tutorial uses separate regions for 0–3.
The sample grid evaluates the 60 nonzero squared distances.

In [ ]:
distance_squares = integer_distances**2
regions = []
initial = []
updated = []
for value in distance_squares:
    power = int(value).bit_length() - 1
    upper = int(value >= 1.5 * 2**power)
    estimate = cubic(value, upper, power)
    regions.append(upper)
    initial.append(estimate)
    updated.append(newton(value, estimate, upper))
expected = 1 / integer_distances
# Independently captured from the unchanged, pinned notebook.
np.testing.assert_allclose(max(abs(initial - expected)), 5.8675108801198306e-5, rtol=0, atol=3e-16)
np.testing.assert_allclose(max(abs(updated - expected)), 2.5821014215665627e-9, rtol=0, atol=3e-16)
print(f"Maximum cubic error: {max(abs(initial - expected)):.12g}")
print(f"Maximum corrected error: {max(abs(updated - expected)):.12g}")

fig, axis = plt.subplots(figsize=(6.3, 3.5), layout="constrained")
axis.plot(distance_squares, expected, label="Exact")
for region in range(2):
    select = np.array(regions) == region
    axis.scatter(distance_squares[select], np.array(initial)[select], s=12, label=f"Region {region + 1}")
axis.set(xlabel="Integer squared distance", ylabel="Inverse integer distance")
axis.legend()
show_figure(fig)

distance_bits = 2 * 6 + 2
intervals = [(value, value) for value in range(4)]
for power in range(2, distance_bits):
    start = 1 << power
    half = 1 << (power - 1)
    intervals.extend([(start, start + half - 1), (start + half, 2 * start - 1)])
assert intervals[0][0] == 0 and intervals[-1][1] == 2**distance_bits - 1
assert all(a[1] + 1 == b[0] for a, b in zip(intervals, intervals[1:]))
assert len(intervals) == 28
print("Region intervals:", intervals)
print("Modeled lookup Toffoli equivalents:", len(intervals) - 2)

## Trotter term models

Kinetic energy computes a sum of squares and phases it. The Coulomb term
iterates unordered pairs, computes their distance and uses a four-bank QROM,
a cubic evaluation and one Newton update. The native append path clears its
workspace. This section uses the two-electron, 21-point model from the
locked tutorial.

In [ ]:
from examples.algorithms.grid_hamiltonian_resources import main as print_budgets
from flagquantum.algorithms.primitives.grid_inverse_sqrt import (
    source_grid_inverse_sqrt_newton_cost,
    source_grid_inverse_sqrt_polynomial_cost,
)
from flagquantum.algorithms.primitives.grid_kinetic import (
    source_grid_kinetic_cost,
    source_grid_kinetic_signature,
)
from flagquantum.algorithms.primitives.grid_potential_resource_model import (
    source_grid_pair_potential_cost,
    source_grid_potential_cost,
)
from flagquantum.drawer.call_graph import export_modeled_call_graph_dot, render_call_graph_svg
from flagquantum.drawer.signature_graph import export_signature_graph_dot


def show_dot(dot):
    display(SVG(data=render_call_graph_svg(dot).decode()))


show_dot(export_signature_graph_dot(source_grid_kinetic_signature(2, 21), "KineticEnergy", pretty=True))
show_dot(export_modeled_call_graph_dot(source_grid_potential_cost(2, 21), max_depth=1))
print_budgets()

## Comparison with the paper

Show the polynomial and Newton call graphs and the four-electron kinetic and
potential graphs. Render a separate pair graph so its arithmetic and lookup
costs can be examined directly. The view hides bookkeeping allocations and releases. The numeric report includes the full model.

In [ ]:
from flagquantum.compiler._component_trace.call_graph import modeled_call_graph
from flagquantum.compiler._component_trace.gate_budget import legacy_gate_budget
from flagquantum.compiler._component_trace.resource_model import estimate_component_resources
from flagquantum.compiler._component_trace.terminal_t_cost import rotation_t_cost
from flagquantum.drawer.call_graph import export_call_graph_dot


def show_comparison(model):
    calls = modeled_call_graph(
        model,
        generalizer=lambda child: None if child.name in {"Allocate", "Free"} else child,
    )
    show_dot(export_call_graph_dot(calls.root, calls.direct_calls, title=lambda node: node.name))


poly = source_grid_inverse_sqrt_polynomial_cost(14, 15, 24, legacy_shims=True)
newton_model = source_grid_inverse_sqrt_newton_cost(14, 15, 24, legacy_shims=True)
pair = source_grid_pair_potential_cost(6, legacy_shims=True)
for model in (poly, newton_model, source_grid_kinetic_cost(4, 21),
              source_grid_potential_cost(4, 21), pair):
    show_comparison(model)


def legacy(model):
    return legacy_gate_budget(estimate_component_resources(model).gates)


comparison = (
    ("Polynomial", legacy(poly)["t"], 4 * (3 * 15**2 + 45)),
    ("Newton", legacy(newton_model)["t"], 4 * (2136 - 3 * 15**2 - 45)),
    ("Pair without synthesized rotations", legacy(pair)["t"], 4 * (2136 + 3 * 6**2 - 6 - 1 + 4 * 6 + 2)),
)
display(Markdown("\n".join((
    "| Stage | Legacy T before rotation synthesis | Paper T |",
    "| --- | ---: | ---: |",
    *(f"| {stage} | {actual} | {paper} |" for stage, actual, paper in comparison),
))))
pair_budget = legacy(pair)
rotation_overhead = pair_budget["rotations"] * rotation_t_cost(1e-11)
assert rotation_overhead == 1248
assert tuple((actual, paper) for _, actual, paper in comparison) == (
    (2676, 2880), (5768, 5664), (9176, 9052)
)
assert pair_budget["t"] - rotation_overhead == 7928
print("Pair rotation overhead at 1e-11:", rotation_overhead)
print("Pair legacy T after synthesis:", pair_budget["t"] + rotation_overhead)
print("Literal original notebook subtraction:", pair_budget["t"] - rotation_overhead)

## Budget conventions

The QEC resource table converts logical categories to T equivalents.
The legacy count lists T and rotations separately. At tolerance (10^{-11}),
rotation synthesis contributes 52 T per rotation to this model. The locked
tutorial subtracts that overhead from a count whose rotations are already
separate. The explicit subtraction above reproduces the original display;
the pair comparison uses the legacy T count before synthesis.

## Fixed-point representation

A fractional $b_r$-bit word encodes $\sum_{j=0}^{b_r-1}a_j 2^{-j-1}$.
An unsigned $b_i$-bit integer word encodes $\sum_{j=0}^{b_i-1}b_j 2^{b_i-j-1}$.
Both list the most significant bit first. These encodings determine the
scale factors for the multiplication, squaring and additive work in the
cubic and Newton stages. The integers used in the numerical plots are
distance addresses, while the estimate stored by the circuit is fixed point.